# Challenge 3 — Weather Agent with Subagents

**Author:** Ayesha Shafquat  
**Date:** August 20, 2026


**Environment Setup — Cloud Authentication**

In [1]:
!bash <(curl -sSL https://storage.googleapis.com/cloud-samples-data/adc/setup_adc.sh)

   Google Cloud Model API & Gemini: ADC setup script
✅ gcloud CLI found at: /root/google-cloud-sdk/bin/gcloud

--- Project Setup ---
Enter your Google Cloud Project ID (NOT the name).
Project ID: qwiklabs-gcp-02-11cee3bd1883

--- Authenticating ---
Authorizing Application Default Credentials (ADC)...

You are running on a Google Compute Engine virtual machine.
The service credentials associated with this virtual machine
will automatically be used by Application Default
Credentials, so it is not necessary to use this command.

If you decide to proceed anyway, your user credentials may be visible
to others with access to this virtual machine. Are you sure you want
to authenticate with your personal account?

Do you want to continue (Y/n)?  y

Go to the following link in your browser, and complete the sign-in prompts:

    https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=764086051850-6qr4p6gpi6hn506pt8ejuq83di341hur.apps.googleusercontent.com&redirect_uri=https%3A%2F

**Environment Setup — Install Required Packages**

In [2]:
!pip install -q google-adk requests

**Environment Setup - Configure Vertex AI**

In [3]:
import os
import google.auth

LOCATION = "us-central1"

try:
    _, project_id = google.auth.default()
except google.auth.exceptions.DefaultCredentialsError:
    project_id = None

if not project_id:
    project_id = input("Could not auto-detect a GCP project. Enter your project ID: ").strip()

os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "TRUE"
os.environ["GOOGLE_CLOUD_PROJECT"] = project_id
os.environ["GOOGLE_CLOUD_LOCATION"] = LOCATION

print(f"Using Vertex AI project '{project_id}' in '{LOCATION}'.")

Using Vertex AI project 'qwiklabs-gcp-02-11cee3bd1883' in 'us-central1'.


**Google Maps Geocoding API Tool**

In [4]:
import requests
from typing import Dict, Optional, List
from getpass import getpass

GOOGLE_MAPS_API_KEY = getpass("Enter your Google Maps API key: ")

def get_lat_lon(city: str, state: str) -> Optional[Dict[str, float]]:
    """Use the Google Maps Geocoding API to convert city and state to latitude and longitude.

    Args:
        city: City name, e.g. "Pittsburgh"
        state: State abbreviation or full name, e.g. "PA"

    Returns:
        Optional[Dict[str, float]]: {'lat': ..., 'lon': ...} or None on failure
    """
    address = f"{city}, {state}, USA"
    url = "https://maps.googleapis.com/maps/api/geocode/json"
    params = {
        "address": address,
        "key": GOOGLE_MAPS_API_KEY
    }
    try:
        resp = requests.get(url, params=params, timeout=10).json()
    except requests.RequestException:
        return None

    if resp.get("status") != "OK" or not resp.get("results"):
        print(f"Geocoding failed: {resp.get('status')}")
        return None

    location = resp["results"][0]["geometry"]["location"]
    return {"lat": location["lat"], "lon": location["lng"]}

Enter your Google Maps API key: ··········


**Test - Geocode a U.S. City**

In [5]:
print(get_lat_lon("Annapolis", "MD"))

{'lat': 38.9764364, 'lon': -76.489642}


**National Weather Service API Tool**

In [6]:
NWS_USER_AGENT = "adk-weather-workshop-ayesha"

def get_extended_weather_forecast(lat: float, lon: float) -> Optional[List[Dict[str, str]]]:
    """Fetch the extended weather forecast from the U.S. National Weather Service API.

    Args:
        lat (float): Latitude of the location.
        lon (float): Longitude of the location.

    Returns:
        Optional[List[Dict[str, str]]]: List of forecast periods, or None on failure.
    """
    headers = {"User-Agent": NWS_USER_AGENT}
    points_url = f"https://api.weather.gov/points/{lat},{lon}"
    try:
        points_response = requests.get(points_url, headers=headers, timeout=10)
        points_response.raise_for_status()
        forecast_url = points_response.json()["properties"]["forecast"]

        forecast_response = requests.get(forecast_url, headers=headers, timeout=10)
        forecast_response.raise_for_status()
        periods = forecast_response.json()["properties"]["periods"]
    except (requests.RequestException, KeyError):
        return None

    return [
        {
            "name": p["name"],
            "temperature": str(p["temperature"]),
            "temperatureUnit": p["temperatureUnit"],
            "shortForecast": p["shortForecast"],
            "windSpeed": p["windSpeed"],
            "windDirection": p["windDirection"],
        }
        for p in periods
    ]

**Test - Edison, Location & Temperature**

In [7]:
location = get_lat_lon("Edison", "NJ")
print(location)

forecast = get_extended_weather_forecast(location["lat"], location["lon"])
print(forecast)

{'lat': 40.5168636, 'lon': -74.40628250000002}
[{'name': 'Overnight', 'temperature': '64', 'temperatureUnit': 'F', 'shortForecast': 'Mostly Cloudy', 'windSpeed': '0 mph', 'windDirection': ''}, {'name': 'Saturday', 'temperature': '77', 'temperatureUnit': 'F', 'shortForecast': 'Chance Rain Showers', 'windSpeed': '0 to 5 mph', 'windDirection': 'E'}, {'name': 'Saturday Night', 'temperature': '65', 'temperatureUnit': 'F', 'shortForecast': 'Slight Chance Showers And Thunderstorms', 'windSpeed': '5 mph', 'windDirection': 'E'}, {'name': 'Sunday', 'temperature': '83', 'temperatureUnit': 'F', 'shortForecast': 'Chance Showers And Thunderstorms', 'windSpeed': '0 to 10 mph', 'windDirection': 'S'}, {'name': 'Sunday Night', 'temperature': '61', 'temperatureUnit': 'F', 'shortForecast': 'Chance Showers And Thunderstorms then Mostly Clear', 'windSpeed': '5 mph', 'windDirection': 'W'}, {'name': 'Monday', 'temperature': '81', 'temperatureUnit': 'F', 'shortForecast': 'Sunny', 'windSpeed': '5 to 10 mph', 'w

**Weather Agent**

In [22]:
from google.adk.agents import Agent

WEATHER_AGENT_INSTRUCTIONS = """
You are Aisha weather specialist.

When the user asks for weather in a U.S. city:
1. Use get_lat_lon to get the latitude and longitude.
2. Use get_extended_weather_forecast to get the NWS forecast.
3. Return a short, clear weather summary.

Only handle weather-related requests.
"""

weather_agent = Agent(
    name="weather_agent",
    model="gemini-2.5-flash",
    description="Specialist for U.S. weather forecasts.",
    instruction=WEATHER_AGENT_INSTRUCTIONS,
    tools=[
        get_lat_lon,
        get_extended_weather_forecast,
    ],
)

print("Weather agent created.")


Weather agent created.


**Search Agent**

In [23]:
def show_search_agent(
    callback_context,
    llm_request
):
    print("[DELEGATION] search_agent is handling this request")
    return None

search_agent = Agent(
    name="search_agent",
    model="gemini-2.5-flash",
    description="Specialist for current information using Google Search.",
    instruction=SEARCH_AGENT_INSTRUCTIONS,
    tools=[google_search],
    before_model_callback=show_search_agent,
)

print("Search agent created.")




Search agent created.


**Coordinator/Root Agent**

In [24]:
from google.adk.agents import LlmAgent
from google.adk.tools import agent_tool

ROOT_AGENT_INSTRUCTIONS = """
You are the coordinating agent.

- For weather questions, delegate to weather_agent.
- For current information, news, events, or web research,
  use search_agent.
- If a request needs both weather and web information,
  use both specialists as appropriate.
"""

root_agent = LlmAgent(
    name="root_agent",
    model="gemini-2.5-flash",
    description="Coordinates weather and search requests.",
    instruction=ROOT_AGENT_INSTRUCTIONS,

    # Search agent is used as a tool
    tools=[
        agent_tool.AgentTool(agent=search_agent)
    ],

    # Weather agent remains a sub-agent
    sub_agents=[
        weather_agent
    ],
)

print("Root agent created.")

Root agent created.


**ADK App**

In [25]:
import vertexai
from vertexai.preview import reasoning_engines

vertexai.init(
    project=project_id,
    location=LOCATION,
)

app = reasoning_engines.AdkApp(
    agent=root_agent,
)

print("Multi-agent app is ready.")


Multi-agent app is ready.


**Helper**

In [26]:
def run_agent_test(message: str) -> None:
    """Run a test and print events showing which agent handled the request."""
    user_id = "challenge3-user"

    session = app.create_session(
        user_id=user_id,
    )

    print(f"USER: {message}\n")

    for event in app.stream_query(
        user_id=user_id,
        session_id=session["id"],
        message=message,
    ):
        author = event.get("author", "unknown")
        print(f"EVENT FROM: {author}")

        content = event.get("content", {})

        for part in content.get("parts", []):
            if "functionCall" in part:
                print(f"  TOOL CALL: {part['functionCall']}")

            if "text" in part and part["text"]:
                text = part["text"].strip()
                print(f"  TEXT: {text[:500]}")

        print()


**Tests**

In [13]:
run_agent_test(
    "What's the weather in Edison, New Jersey?"
)


/usr/local/lib/python3.12/dist-packages/vertexai/preview/reasoning_engines/templates/adk.py:966: UserWarning: [EXPERIMENTAL] InMemoryCredentialService: This feature is experimental and may change or be removed in future versions without notice. It may introduce breaking changes at any time.
  self._tmpl_attrs["credential_service"] = InMemoryCredentialService()
/usr/local/lib/python3.12/dist-packages/google/adk/auth/credential_service/in_memory_credential_service.py:33: UserWarning: [EXPERIMENTAL] BaseCredentialService: This feature is experimental and may change or be removed in future versions without notice. It may introduce breaking changes at any time.
  super().__init__()


USER: What's the weather in Edison, New Jersey?



/usr/local/lib/python3.12/dist-packages/google/adk/tools/function_tool.py:95: UserWarning: [EXPERIMENTAL] feature FeatureName.JSON_SCHEMA_FOR_FUNC_DECL is enabled.
  build_function_declaration(


EVENT FROM: root_agent

EVENT FROM: root_agent

EVENT FROM: weather_agent

EVENT FROM: weather_agent

EVENT FROM: weather_agent

EVENT FROM: weather_agent

EVENT FROM: weather_agent
  TEXT: The weather in Edison, New Jersey: Overnight will be mostly cloudy with a temperature of 64°F. Saturday has a chance of rain showers and a high of 77°F. There's a chance of showers and thunderstorms on Sunday with a high of 83°F. Monday and Tuesday are expected to be sunny with temperatures around 80-81°F. The rest of the week shows chances of rain showers and thunderstorms.



In [27]:
run_agent_test(
    "What are the latest updates about Bitcoin?"
)


/usr/local/lib/python3.12/dist-packages/vertexai/preview/reasoning_engines/templates/adk.py:966: UserWarning: [EXPERIMENTAL] InMemoryCredentialService: This feature is experimental and may change or be removed in future versions without notice. It may introduce breaking changes at any time.
  self._tmpl_attrs["credential_service"] = InMemoryCredentialService()
/usr/local/lib/python3.12/dist-packages/google/adk/auth/credential_service/in_memory_credential_service.py:33: UserWarning: [EXPERIMENTAL] BaseCredentialService: This feature is experimental and may change or be removed in future versions without notice. It may introduce breaking changes at any time.
  super().__init__()


USER: What are the latest updates about Bitcoin?

EVENT FROM: root_agent

[DELEGATION] search_agent is handling this request
EVENT FROM: root_agent

EVENT FROM: root_agent
  TEXT: Bitcoin has recently seen a surge, with its price reaching $78,041.11, a 4.26% increase in the last 24 hours and a 23.80% increase over the past 7 days. This rise is partly due to a treasury buyback, which has triggered a Bitcoin surge as a "debasement trade" re-emerges.

Additionally, Bitcoin and Ethereum Exchange Traded Funds (ETFs) have attracted $827 million, suggesting a wider rally in the cryptocurrency market. For more detailed news and live updates, you can refer to resources like The Bl

